# Company Enrichment System

**Relu Consultancy AI & Automation Developer Hiring Challenge**

This notebook calls the Company Enrichment API to scrape and AI-enrich any company website.

**Output schema per company:**
```
website_name        — domain
company_name        — scraped company name
address             — physical address (if found)
mobile_number       — phone number (if found)
mail                — list of emails found
core_service        — AI: what they sell
target_customer     — AI: who their buyer is
probable_pain_point — AI: problem they solve
outreach_opener     — AI: personalized cold-email opener
```

In [ ]:
# Install dependencies
!pip install requests --quiet

In [ ]:
import requests
import json
import os
from datetime import datetime

# ── Configuration ─────────────────────────────────────────────────────────────
# Update BASE_URL to the live deployment URL after deployment.
# Example: BASE_URL = "https://company-enrichment.up.railway.app"
BASE_URL = "http://localhost:3000"

RESULTS_FILE = "results.json"
TIMEOUT_SECONDS = 60  # enrichment can take up to 30s for slow sites

# Schema keys — all must be present in every valid response
SCHEMA_KEYS = [
    "website_name", "company_name", "address", "mobile_number",
    "mail", "core_service", "target_customer",
    "probable_pain_point", "outreach_opener"
]

print(f"API base URL: {BASE_URL}")
print(f"Results will be saved to: {os.path.abspath(RESULTS_FILE)}")

In [ ]:
def enrich_company(url, company_name=""):
    """
    Enrich a company by URL.

    Args:
        url (str): Full company website URL, e.g. "https://stripe.com"
        company_name (str): Optional name hint for the AI (can be empty)

    Returns:
        dict: Strict schema dict on success, None on failure.

    Schema keys:
        website_name, company_name, address, mobile_number, mail[],
        core_service, target_customer, probable_pain_point, outreach_opener
    """
    try:
        print(f"\n→ Enriching: {url}")

        response = requests.post(
            f"{BASE_URL}/enrich",
            json={"companyName": company_name, "url": url},
            timeout=TIMEOUT_SECONDS,
        )

        data = response.json()

        # API-level error (validation failure, scraping failure, etc.)
        if not response.ok:
            err = data.get("error") or (
                data.get("errors", [{}])[0].get("message", "Unknown error")
            )
            print(f"  ✗ API error ({response.status_code}): {err}")
            return None

        # Validate strict schema
        missing = [k for k in SCHEMA_KEYS if k not in data]
        if missing:
            print(f"  ✗ Schema incomplete — missing keys: {missing}")
            return None

        print(f"  ✓ Success — {data.get('company_name') or data.get('website_name')}")
        return data

    except requests.exceptions.ConnectionError:
        print(f"  ✗ Cannot connect to API at {BASE_URL}. Is the server running?")
        return None
    except requests.exceptions.Timeout:
        print(f"  ✗ Request timed out after {TIMEOUT_SECONDS}s.")
        return None
    except Exception as e:
        print(f"  ✗ Unexpected error: {e}")
        return None


print("enrich_company() ready.")

In [ ]:
def save_results(result, filepath=RESULTS_FILE):
    """
    Upsert a result into the local results JSON file.
    Matches by website_name (domain). Replaces existing entry, appends if new.
    """
    if result is None:
        return

    # Load existing
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            existing = json.load(f)
        if not isinstance(existing, list):
            existing = []
    except (FileNotFoundError, json.JSONDecodeError):
        existing = []

    # Upsert by website_name
    domain = result.get("website_name")
    idx = next((i for i, r in enumerate(existing) if r.get("website_name") == domain), -1)
    if idx >= 0:
        existing[idx] = result
    else:
        existing.append(result)

    with open(filepath, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)

    print(f"  Saved to {os.path.abspath(filepath)} ({len(existing)} record(s))")


print("save_results() ready.")

## Demo — Enrich Example Companies

Runs `enrich_company(url)` on three real company URLs and saves the output.

In [ ]:
# Demo URLs — replace or extend as needed
DEMO_URLS = [
    "https://stripe.com",
    "https://vercel.com",
    "https://netlify.com",
]

demo_results = []

for url in DEMO_URLS:
    result = enrich_company(url)
    if result:
        save_results(result)
        demo_results.append(result)

print(f"\nCompleted: {len(demo_results)}/{len(DEMO_URLS)} enriched successfully.")

## Results

In [ ]:
# Display enriched results
for r in demo_results:
    print("─" * 60)
    print(f"Domain         : {r.get('website_name', '')}")
    print(f"Company        : {r.get('company_name', '')}")
    print(f"Phone          : {r.get('mobile_number', '') or '—'}")
    print(f"Email(s)       : {', '.join(r.get('mail', [])) or '—'}")
    print(f"Address        : {r.get('address', '') or '—'}")
    print(f"Core Service   : {r.get('core_service', '') or '—'}")
    print(f"Target Customer: {r.get('target_customer', '') or '—'}")
    print(f"Pain Point     : {r.get('probable_pain_point', '') or '—'}")
    print(f"Outreach Opener: {r.get('outreach_opener', '') or '—'}")
print("─" * 60)

In [ ]:
# Show raw JSON for the first result (strict schema validation)
if demo_results:
    print("Raw JSON (first result):")
    print(json.dumps(demo_results[0], indent=2, ensure_ascii=False))

## Single Lookup

Use this cell to enrich any company on demand.

In [ ]:
# Enrich a single company — change the URL below
url = "https://linear.app"

data = enrich_company(url)
if data:
    save_results(data)
    print(json.dumps(data, indent=2, ensure_ascii=False))

## Error Handling Demo

Shows graceful failure — no crash on bad/unreachable URLs.

In [ ]:
# These should fail gracefully — no exceptions raised
bad_urls = [
    "https://this-domain-absolutely-does-not-exist-xyz.com",
    "not-a-url",
]

for bad_url in bad_urls:
    result = enrich_company(bad_url)
    print(f"  Result for {bad_url!r}: {result}")  # None expected — no crash

print("\nAll error cases handled without crashing.")

## Saved results.json

In [ ]:
# Read and display the saved results file
try:
    with open(RESULTS_FILE, "r", encoding="utf-8") as f:
        saved = json.load(f)
    print(f"results.json — {len(saved)} record(s)")
    print(json.dumps(saved, indent=2, ensure_ascii=False))
except FileNotFoundError:
    print("results.json not yet created — run the demo cells above first.")